# Demo 1 — Event Hub Consumer to Bronze

This notebook reads the crypto JSON events from Azure Event Hubs with Spark Structured Streaming and writes them into a permanent Bronze Delta table.

It demonstrates:
- connecting Spark to Event Hubs through its Kafka-compatible endpoint
- reading Event Hub events as a stream
- parsing the JSON body with an explicit schema
- retaining Event Hub/Kafka metadata
- writing incrementally to a Unity Catalog Delta table
- using a checkpoint so previously processed events are not loaded again

The producer has already sent 30 events. This notebook uses `startingOffsets = earliest` on the first run so those existing events can be consumed.


## 1. Load shared configuration

The Event Hub name, secret settings, checkpoint path, and Bronze table name come from `config/00_config`.


In [0]:
%run ../config/00_config


## 2. Import required Spark functions and data types


In [0]:
import re

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType,
)


## 3. Retrieve the Event Hub connection string

The connection string is stored in Databricks secrets. Its value is used for authentication but is never printed.


In [0]:
try:
    eventhub_connection_string = dbutils.secrets.get(
        scope=eventhub_secret_scope,
        key=eventhub_secret_key,
    )
except Exception as exc:
    raise RuntimeError(
        "Could not retrieve the Event Hub connection string. "
        f"Check secret scope '{eventhub_secret_scope}' and "
        f"secret key '{eventhub_secret_key}'."
    ) from exc

if not eventhub_connection_string:
    raise RuntimeError("The Event Hub connection string is empty.")

print("Event Hub connection string retrieved successfully.")


## 4. Build the Event Hubs Kafka connection settings

Azure Event Hubs exposes a Kafka-compatible endpoint on port `9093`.

The namespace hostname is extracted from the connection string, for example:

`Endpoint=sb://my-namespace.servicebus.windows.net/;...`

becomes:

`my-namespace.servicebus.windows.net:9093`


In [0]:
endpoint_match = re.search(
    r"Endpoint=sb://([^/;]+)",
    eventhub_connection_string,
    flags=re.IGNORECASE,
)

if not endpoint_match:
    raise ValueError(
        "Could not extract the Event Hubs namespace endpoint "
        "from the connection string."
    )

eventhub_namespace_host = endpoint_match.group(1)
kafka_bootstrap_servers = f"{eventhub_namespace_host}:9093"

escaped_connection_string = (
    eventhub_connection_string
    .replace("\\", "\\\\")
    .replace('"', '\\"')
)

kafka_sasl_jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.'
    'PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{escaped_connection_string}";'
)

print(f"Kafka bootstrap server: {kafka_bootstrap_servers}")
print(f"Event Hub topic: {eventhub_name}")
print(f"Checkpoint: {crypto_ticks_checkpoint_path}")
print(f"Bronze table: {streaming_bronze_table}")


## 5. Define the initial JSON event schema

This schema matches the producer's first event version:

- `event_id`
- `symbol`
- `price_usd`
- `event_time`
- `producer_id`
- `source_system`

New fields will be introduced later in the schema-evolution notebook.


In [0]:
price_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("symbol", StringType(), False),
    StructField("price_usd", DoubleType(), False),
    StructField("event_time", StringType(), False),
    StructField("producer_id", StringType(), False),
    StructField("source_system", StringType(), False),
])


## 6. Create the raw Event Hub stream

Event Hubs appears to Spark as a Kafka topic:

- the Event Hub name is the Kafka topic
- `value` contains the JSON event body
- Kafka metadata includes partition, offset, and broker timestamp

`startingOffsets = earliest` applies only when no checkpoint exists. After the first run, the checkpoint controls where processing resumes.


In [0]:
raw_eventhub_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", eventhub_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", kafka_sasl_jaas_config)
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "30000")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

print("Raw Event Hub stream created successfully.")


## 7. Parse the JSON events and add ingestion metadata

The event body is converted from binary to text and parsed using the explicit schema.

The Bronze output also keeps:
- Event Hub partition
- Event Hub offset
- Event Hub enqueue/Kafka timestamp
- raw JSON body
- ingestion timestamp


In [0]:
parsed_stream_df = (
    raw_eventhub_stream_df
    .withColumn("raw_json", F.col("value").cast("string"))
    .withColumn(
        "parsed_event",
        F.from_json(F.col("raw_json"), price_event_schema),
    )
    .select(
        F.col("parsed_event.event_id").alias("event_id"),
        F.col("parsed_event.symbol").alias("symbol"),
        F.col("parsed_event.price_usd").alias("price_usd"),
        F.to_timestamp(
            F.col("parsed_event.event_time")
        ).alias("event_time"),
        F.col("parsed_event.producer_id").alias("producer_id"),
        F.col("parsed_event.source_system").alias("source_system"),
        F.col("partition").alias("eventhub_partition"),
        F.col("offset").alias("eventhub_offset"),
        F.col("timestamp").alias("eventhub_enqueued_at"),
        F.col("raw_json"),
        F.current_timestamp().alias("ingested_at"),
    )
)

print("JSON parsing transformation prepared.")


## 8. Separate valid and invalid events

A valid event must have:
- `event_id`
- `symbol`
- `price_usd`
- a successfully parsed `event_time`

For this first version, invalid events are displayed through a temporary debug stream rather than written to a separate table.


In [0]:
valid_stream_df = parsed_stream_df.filter(
    F.col("event_id").isNotNull()
    & F.col("symbol").isNotNull()
    & F.col("price_usd").isNotNull()
    & F.col("event_time").isNotNull()
)

invalid_stream_df = parsed_stream_df.filter(
    F.col("event_id").isNull()
    | F.col("symbol").isNull()
    | F.col("price_usd").isNull()
    | F.col("event_time").isNull()
)


## 9. Write valid events to the streaming Bronze table

The stream writes in append mode to:

`dbr_dev.parvinbadalov.demo1_crypto_ticks_bronze`

`availableNow=True` processes all events currently available and then stops. This makes the demo easy to rerun and suitable for a workflow task.

The checkpoint prevents the same Event Hub offsets from being written again.


In [0]:
bronze_query = (
    valid_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", crypto_ticks_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(streaming_bronze_table)
)

bronze_query.awaitTermination()

print("Event Hub Bronze ingestion completed.")


## 10. Validate the Bronze table

The first successful run should normally contain the 30 events already sent by the producer:

- 10 BTC events
- 10 ETH events
- 10 SOL events


In [0]:
streaming_bronze_df = spark.table(streaming_bronze_table)

summary_df = (
    streaming_bronze_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_event_time"),
        F.max("event_time").alias("last_event_time"),
        F.min("price_usd").alias("minimum_price"),
        F.max("price_usd").alias("maximum_price"),
    )
    .orderBy("symbol")
)

display(summary_df)

total_events = streaming_bronze_df.count()
distinct_event_ids = (
    streaming_bronze_df
    .select("event_id")
    .distinct()
    .count()
)

print(f"Streaming Bronze table: {streaming_bronze_table}")
print(f"Total events: {total_events}")
print(f"Distinct event IDs: {distinct_event_ids}")

if total_events != distinct_event_ids:
    print(
        "Warning: duplicate event IDs exist. "
        "They will be handled in the Silver transformation."
    )
else:
    print("No duplicate event IDs found.")


## 11. Inspect Event Hub metadata

These columns prove that the records came through Event Hubs and show their partition and offset positions.


In [0]:
display(
    streaming_bronze_df
    .select(
        "event_id",
        "symbol",
        "price_usd",
        "event_time",
        "eventhub_partition",
        "eventhub_offset",
        "eventhub_enqueued_at",
        "ingested_at",
    )
    .orderBy("eventhub_partition", "eventhub_offset")
)


## 12. Rerun behavior

Run this notebook again without sending new producer events.

Expected result:
- the checkpoint resumes from the last processed offsets
- no old Event Hub events are appended again
- the Bronze row count stays unchanged

Then run the producer again and rerun this consumer:
- only the newly produced events are added

## Next notebook

`streaming/06_schema_evolution_demo.ipynb`

That notebook will add new fields to the producer events and evolve the streaming Bronze table schema.
